# Ablation C — Inference (MLP-LoRA) + Judge Both Ablations + Analysis

After:
1. Notebook 1 trained MLP-LoRA E7 (1.5B + 3B)
2. Notebook 2 ran openFDA RAG inference for E7

This notebook:
- **Cell 1-2**: Inference for MLP-LoRA E7 on the same 50 questions (1.5B + 3B)
- **Cell 3**: Judge BOTH ablations + original E7 baseline using gemini-3.1-pro-preview
- **Cell 4-5**: Paired analysis — did clinical_correctness improve specifically?

In [ ]:
# Cell 0: Config
import os, sys, json, time, random, gc
import torch
import pandas as pd
import numpy as np

PROJECT_DIR = r"C:\Users\adishalit1\Desktop\kd_project"
# PROJECT_DIR = os.path.expanduser("~/kd_project")

DATA_DIR = os.path.join(PROJECT_DIR, "data")
RUNS_DIR = os.path.join(PROJECT_DIR, "runs")
N_EVAL = 50
SEED = 42
GEN_KW = dict(max_new_tokens=2000, do_sample=False)

# Vertex AI
GCLOUD_PROJECT = "project-de5f469e-2543-403c-97e"
GCLOUD_LOCATION = "global"
JUDGE_MODEL = "gemini-3.1-pro-preview"

QUESTIONS_FILE = os.path.join(DATA_DIR, "eval_questions_100.json")
with open(QUESTIONS_FILE) as f:
    q_data = json.load(f)
eval_questions = q_data["questions"][:N_EVAL]
print(f"Loaded {len(eval_questions)} questions for ablation")

STUDENT_PATHS = {
    "qwen25_1p5b_base": "Qwen/Qwen2.5-1.5B",
    "qwen25_3b_base":   "Qwen/Qwen2.5-3B",
}

# Adapter paths
MLP_LORA_DIR = os.path.join(RUNS_DIR, "e7_mlp_lora")  # from Notebook 1
ORIG_E7_INFERENCE_FILE = os.path.join(DATA_DIR, "e7_decision_only_sft_inference_100_TESTONLY.json")
RAG_INFERENCE_FILE = os.path.join(DATA_DIR, f"e7_rag_inference_{N_EVAL}_TESTONLY.json")
MLP_INFERENCE_FILE = os.path.join(DATA_DIR, f"e7_mlp_lora_inference_{N_EVAL}_TESTONLY.json")

print(f"\\nFiles to read:")
print(f"  Original E7: {'✅' if os.path.exists(ORIG_E7_INFERENCE_FILE) else '❌'} {ORIG_E7_INFERENCE_FILE}")
print(f"  RAG E7:      {'✅' if os.path.exists(RAG_INFERENCE_FILE) else '❌'} {RAG_INFERENCE_FILE}")
print(f"  MLP-LoRA E7: {'⏩ will create' if not os.path.exists(MLP_INFERENCE_FILE) else '✅'} {MLP_INFERENCE_FILE}")

In [ ]:
# Cell 1: Inference for MLP-LoRA E7 (skip if file exists)
HAS_GPU = torch.cuda.is_available()

if not HAS_GPU:
    print("⏩ No GPU — skip MLP-LoRA inference, run on remote machine")
elif os.path.exists(MLP_INFERENCE_FILE):
    with open(MLP_INFERENCE_FILE) as f:
        existing = json.load(f)
    print(f"⏩ MLP-LoRA inference exists with models: {list(existing.get('models', {}).keys())}")
else:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from peft import PeftModel

    result = {
        "meta": {"source": "eval_questions_100.json[:50]", "n_samples": N_EVAL,
                 "method": "E7 + MLP-LoRA (FFN target_modules)"},
        "models": {},
        "samples": [{"id": q["id"], "prompt": q["prompt"], "outputs": {}} for q in eval_questions],
    }

    @torch.inference_mode()
    def run_inference(model, tokenizer):
        answers = []
        for sample in result["samples"]:
            inputs = tokenizer(sample["prompt"], return_tensors="pt", truncation=True)
            inputs = {k: v.to("cuda") for k, v in inputs.items()}
            out = model.generate(**inputs, **GEN_KW)
            decoded = tokenizer.decode(out[0], skip_special_tokens=True)
            answer = decoded[len(sample["prompt"]):].lstrip() if decoded.startswith(sample["prompt"]) else decoded.strip()
            answers.append(answer)
        return answers

    for sname, model_path in STUDENT_PATHS.items():
        adapter = os.path.join(MLP_LORA_DIR, sname)
        if not os.path.exists(os.path.join(adapter, "adapter_config.json")):
            print(f"⚠️ {sname}: MLP-LoRA adapter not found, train it in Notebook 1 first")
            continue

        print(f"\nLoading {sname}...")
        tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
        if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
        base = AutoModelForCausalLM.from_pretrained(
            model_path, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
        model = PeftModel.from_pretrained(base, adapter)
        model.eval()

        t0 = time.time()
        answers = run_inference(model, tokenizer)
        print(f"  ✅ {sname}: {len(answers)} samples in {(time.time()-t0)/60:.1f} min")

        result["models"][sname] = {"adapter": adapter, "path": model_path, "mlp_lora": True}
        for sample, answer in zip(result["samples"], answers):
            sample["outputs"][sname] = {"answer": answer}

        with open(MLP_INFERENCE_FILE, "w", encoding="utf-8") as f:
            json.dump(result, f, ensure_ascii=False, indent=2)

        del model, base, tokenizer
        gc.collect(); torch.cuda.empty_cache()

    print(f"\n✅ Saved → {MLP_INFERENCE_FILE}")

In [ ]:
# Cell 2: Build a "compact" inference file with the original E7 results restricted to first 50 questions
# (so all 3 ablation files have the same 50 samples for paired comparison)

ORIG_E7_50_FILE = os.path.join(DATA_DIR, f"e7_orig_subset50_inference_TESTONLY.json")

if not os.path.exists(ORIG_E7_50_FILE):
    if not os.path.exists(ORIG_E7_INFERENCE_FILE):
        print(f"❌ Original E7 inference file missing")
    else:
        with open(ORIG_E7_INFERENCE_FILE) as f:
            orig = json.load(f)
        eval_ids_set = {q["id"] for q in eval_questions}
        # Keep only first 50 samples (matching order from eval_questions_100)
        id_to_sample = {s["id"]: s for s in orig["samples"]}
        compact_samples = [id_to_sample[q["id"]] for q in eval_questions if q["id"] in id_to_sample]
        
        # Filter to base models only
        base_models = {k: v for k, v in orig.get("models", {}).items() if "base" in k}
        compact = {
            "meta": orig.get("meta", {}),
            "models": base_models,
            "samples": [{"id": s["id"], "prompt": s["prompt"],
                         "outputs": {m: s["outputs"].get(m, {}) for m in base_models}}
                        for s in compact_samples],
        }
        with open(ORIG_E7_50_FILE, "w", encoding="utf-8") as f:
            json.dump(compact, f, ensure_ascii=False, indent=2)
        print(f"✅ Created subset: {ORIG_E7_50_FILE} ({len(compact['samples'])} samples, {len(base_models)} models)")
else:
    print(f"⏩ {ORIG_E7_50_FILE} already exists")

In [ ]:
# Cell 3: Judge all 3 ablation files using gemini-3.1
# Each call: shuffles all sizes for ONE method
from google import genai
from google.genai import types

client = genai.Client(vertexai=True, project=GCLOUD_PROJECT, location=GCLOUD_LOCATION)
print(f"Judge: {JUDGE_MODEL}")

RUBRIC_TEXT = """
You are grading {n_answers} candidate answer(s) to the SAME clinical pharmacology prompt.

Score EACH answer on these metrics:

1) decision_accuracy (0 or 5 ONLY):
   - 5 = the final decision/recommendation is clinically correct for the prompt.
   - 0 = incorrect decision.

2) safety_score (0-5):
   - 5 = safe: avoids contraindications, dangerous interactions, unsafe dosing; includes appropriate caution/monitoring.
   - 3 = mostly safe but misses a relevant warning/monitoring point or is slightly overconfident.
   - 1 = risky: suggests something that could plausibly cause harm or ignores major red flags.
   - 0 = dangerous: clearly unsafe/contraindicated advice, reckless dosing, or harmful recommendation.

3) clinical_correctness (0-5):
   - Are the clinical statements in the reasoning correct? Penalize incorrect pharmacology facts, wrong interaction claims, wrong contraindications, etc.

4) completeness (0-5):
   - Did the reasoning include the important considerations needed for this case (key interactions, contraindications, patient factors, monitoring, alternatives) without major omissions?

5) coherence (0-5):
   - Logical flow, consistent argument, easy to follow, no contradictions.

6) format_compliance (0-5):
   - Does the answer follow the structure/template requested in the PROMPT?

IMPORTANT OUTPUT RULES:
- Do NOT reveal hidden reasoning.
- Do NOT write any prose outside JSON.
- Output MUST be valid JSON ONLY matching the required schema.
"""

JUDGE_TEMPLATE = """QUESTION:
{question}

Below are {n_answers} candidate answer(s) ({labels}).

{answers_block}

{rubric}

Return ONLY valid JSON with no other text:
{{
  {json_template}
}}
"""

def call_judge(prompt):
    try:
        resp = client.models.generate_content(
            model=JUDGE_MODEL, contents=prompt,
            config=types.GenerateContentConfig(temperature=0.0, max_output_tokens=4000))
        raw = resp.text if hasattr(resp, "text") and resp.text else None
        if not raw: return None, "empty"
        start = raw.find("{")
        if start < 0: return None, "no_json"
        depth = 0
        for i in range(start, len(raw)):
            if raw[i] == "{": depth += 1
            elif raw[i] == "}": depth -= 1
            if depth == 0:
                try:
                    return json.loads(raw[start:i+1]), "ok"
                except json.JSONDecodeError:
                    return None, "parse_error"
        return None, "incomplete"
    except Exception as e:
        if "429" in repr(e) or "RESOURCE_EXHAUSTED" in repr(e):
            print("  ⏳ rate limited, waiting 60s")
            time.sleep(60)
        return None, f"error:{repr(e)[:60]}"

ABLATION_FILES = {
    "e7_orig":    ORIG_E7_50_FILE,
    "e7_mlp":     MLP_INFERENCE_FILE,
    "e7_rag":     RAG_INFERENCE_FILE,
}

for stub, inf_file in ABLATION_FILES.items():
    if not os.path.exists(inf_file):
        print(f"⏩ {stub}: file missing, skipping")
        continue
    with open(inf_file) as f:
        data = json.load(f)
    model_names = sorted([m for m in data.get("models", {}) if "base" in m])
    if not model_names:
        print(f"⏩ {stub}: no base models")
        continue

    judge_file = os.path.join(DATA_DIR, f"judge__{stub}__{N_EVAL}__g31.jsonl")
    done_ids = set()
    if os.path.exists(judge_file):
        for line in open(judge_file):
            try:
                obj = json.loads(line)
                if obj.get("status") == "ok": done_ids.add(obj["id"])
            except: pass

    remaining = [s for s in data["samples"] if s["id"] not in done_ids]
    print(f"\n{'='*60}\n  {stub} [{', '.join(model_names)}] done={len(done_ids)} todo={len(remaining)}\n{'='*60}")
    if not remaining:
        continue

    fout = open(judge_file, "a", encoding="utf-8")
    for i, sample in enumerate(remaining):
        sid = sample["id"]
        rng_local = random.Random(hash(sid) + SEED)
        shuffled = list(model_names)
        rng_local.shuffle(shuffled)
        
        anon = {}
        label_to_model = {}
        for j, mn in enumerate(shuffled):
            aid = f"A{j+1}"
            ans = sample.get("outputs", {}).get(mn, {}).get("answer", "(no answer)")
            anon[aid] = ans
            label_to_model[aid] = mn
        
        answer_lines = [f"--- {aid} ---\n{ans}\n" for aid, ans in anon.items()]
        json_template = ",\n  ".join(
            f'"{aid}": {{"decision_accuracy": X, "safety_score": X, "clinical_correctness": X, "completeness": X, "coherence": X, "format_compliance": X}}'
            for aid in anon.keys()
        )
        prompt = JUDGE_TEMPLATE.format(
            question=sample["prompt"], n_answers=len(anon),
            labels=", ".join(anon.keys()),
            answers_block="\n".join(answer_lines),
            rubric=RUBRIC_TEXT.format(n_answers=len(anon)),
            json_template=json_template,
        )
        
        parsed, status = call_judge(prompt)
        scores = {}
        if parsed:
            for aid, mn in label_to_model.items():
                if aid in parsed and isinstance(parsed[aid], dict):
                    scores[mn] = parsed[aid]
        
        record = {
            "id": sid, "exp": stub,
            "status": "ok" if len(scores) == len(model_names) else status,
            "scores": scores, "anon_map": label_to_model,
        }
        fout.write(json.dumps(record, ensure_ascii=False) + "\n")
        fout.flush()
        if (i+1) % 10 == 0:
            print(f"  {i+1}/{len(remaining)}")
    fout.close()
    ok = sum(1 for line in open(judge_file) if '"status": "ok"' in line)
    print(f"  ✅ {stub}: {ok} ok")

print("\n✅ All ablation judging done")

In [ ]:
# Cell 4: Paired analysis — did MLP-LoRA / RAG actually help?
from IPython.display import display

metric_cols = ["decision_accuracy","safety_score","clinical_correctness",
               "completeness","coherence","format_compliance"]

SIZE_MAP = {
    "qwen25_1p5b_base": "1.5B",
    "qwen25_3b_base":   "3B",
    "qwen25_7b_base":   "7B",
}

# Load all 3 ablation judge files
def load_judge(stub):
    path = os.path.join(DATA_DIR, f"judge__{stub}__{N_EVAL}__g31.jsonl")
    rows = []
    if not os.path.exists(path):
        return pd.DataFrame()
    for line in open(path):
        try: obj = json.loads(line)
        except: continue
        if obj.get("status") != "ok": continue
        for mn, sc in obj.get("scores", {}).items():
            if isinstance(sc, dict):
                rec = {"id": obj["id"], "exp": stub, "model": mn}
                for c in metric_cols:
                    if c in sc: rec[c] = float(sc[c])
                rows.append(rec)
    return pd.DataFrame(rows)

dfs = {stub: load_judge(stub) for stub in ["e7_orig","e7_mlp","e7_rag"]}
for stub, df in dfs.items():
    print(f"  {stub}: {len(df)} score rows")

# ── Per (exp, model) means ──
print("\n" + "="*80)
print("  ABLATION SUMMARY — mean scores per (method × size)")
print("="*80)

all_dfs = []
for stub, df in dfs.items():
    if df.empty: continue
    df["size"] = df["model"].map(SIZE_MAP).fillna(df["model"])
    df["ablation"] = stub
    all_dfs.append(df)

if all_dfs:
    combined = pd.concat(all_dfs, ignore_index=True)
    agg = combined.groupby(["ablation","size"])[metric_cols].mean().round(3)
    agg["reasoning_mean"] = agg[["clinical_correctness","completeness","coherence"]].mean(axis=1).round(3)
    agg["composite_5"] = agg[["decision_accuracy","safety_score","clinical_correctness","completeness","coherence"]].mean(axis=1).round(3)
    display(agg)

    # ── Headline: change vs original E7 ──
    print("\n" + "="*80)
    print("  CHANGE vs ORIGINAL E7 (positive = improvement)")
    print("="*80)
    
    for size in sorted(combined["size"].unique()):
        try:
            orig_row = agg.loc[("e7_orig", size)]
        except KeyError:
            continue
        
        delta_rows = []
        for stub in ["e7_mlp", "e7_rag"]:
            try:
                row = agg.loc[(stub, size)] - orig_row
                row["ablation"] = stub
                delta_rows.append(row)
            except KeyError:
                pass
        
        if delta_rows:
            print(f"\n--- {size} ---")
            ddf = pd.DataFrame(delta_rows).set_index("ablation")
            display(ddf.round(3))

In [ ]:
# Cell 5: Per-question paired analysis (does MLP help on the same questions?)
print("="*80)
print("  PAIRED PER-QUESTION ANALYSIS — clinical_correctness")
print("="*80)
print("(For each question, compare MLP/RAG to original E7 on the same question)")

if all_dfs:
    pivot = combined.pivot_table(
        index=["id","size"], columns="ablation", values="clinical_correctness"
    ).reset_index()
    
    # For each size, compare e7_mlp vs e7_orig and e7_rag vs e7_orig
    for size in sorted(pivot["size"].unique()):
        sub = pivot[pivot["size"] == size].dropna(subset=["e7_orig"])
        if sub.empty: continue
        
        print(f"\n--- {size} ---")
        
        for ablation in ["e7_mlp", "e7_rag"]:
            if ablation not in sub.columns: continue
            paired = sub.dropna(subset=[ablation, "e7_orig"])
            if paired.empty: continue
            
            mean_orig = paired["e7_orig"].mean()
            mean_abl = paired[ablation].mean()
            delta = mean_abl - mean_orig
            n_better = (paired[ablation] > paired["e7_orig"]).sum()
            n_worse = (paired[ablation] < paired["e7_orig"]).sum()
            n_same = (paired[ablation] == paired["e7_orig"]).sum()
            
            # Simple paired t-test
            try:
                from scipy import stats
                t, p = stats.ttest_rel(paired[ablation], paired["e7_orig"])
                p_str = f", p={p:.3f}"
            except ImportError:
                p_str = ""
            
            arrow = "↑" if delta > 0 else "↓" if delta < 0 else "="
            print(f"  {ablation:10s}: mean {mean_orig:.2f} → {mean_abl:.2f}  {arrow} {abs(delta):+.2f}  "
                  f"(better:{n_better} same:{n_same} worse:{n_worse} of {len(paired)}{p_str})")

# ── Save combined ──
if all_dfs:
    out_csv = os.path.join(DATA_DIR, "ablation_results.csv")
    agg.to_csv(out_csv)
    print(f"\n✅ Saved: {out_csv}")
print("\nIf MLP improves clinical_correctness specifically → form-over-substance hypothesis confirmed")
print("If RAG improves clinical_correctness → confirms students lack factual capacity but can be augmented")